# Build the Raiders reference corpus

Build and save reference posts and exact anchor occurrences automatically from the reference partition. No manual annotation or approval step is required.

For this experiment, anchor matches are **assumed suitable references**, not validated hate labels. That assumption is recorded in the manifest. This notebook prepares the text and spans; contextual BERT embedding remains a separate step.

## 1. Settings

The existing pilot settings are retained: up to **15 contexts per anchor**, with seed **42**, at most one occurrence per thread and one normalized post text per anchor. Set the cap to `None` for all eligible distinct contexts; the thread/text diversity rules still apply. Rare anchors are never padded.

Choose any category from the shared vocabulary. Change `RUN_NAME` when changing the source, settings or vocabulary. The new output directory is separate from any earlier manual-review files.


In [1]:
from pathlib import Path
import sys

# Supports kernels launched either from the repository root or from src/.
ROOT = Path.cwd()
if ROOT.name == "src":
    ROOT = ROOT.parent
if not (ROOT / "src/utils/vocabulary.py").is_file():
    raise RuntimeError("Open this notebook from the project root or its src directory.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from html import escape
from IPython.display import HTML, display
from utils.vocabulary import anchor_terms_by_hate_category
from utils.reference_review import prepare_references

CATEGORY = "antisemitism"
REFERENCE_PATH = ROOT / "data/raiders/raiders_first_1GB.ndjson"
MAX_REFERENCES_PER_ANCHOR = 15  # None removes the cap, retaining thread/text diversity.
RANDOM_SEED = 42
RUN_NAME = f"{CATEGORY}_v1"
OUTPUT_DIR = ROOT / "data/raiders/reference_corpora" / RUN_NAME

anchors = anchor_terms_by_hate_category[CATEGORY]
print("Reference source:", REFERENCE_PATH)
print("Output folder:", OUTPUT_DIR)


Reference source: /home/armand/MS_IA/IA717/project/Emerging_Coded_Hate_Speech/data/raiders/raiders_first_1GB.ndjson
Output folder: /home/armand/MS_IA/IA717/project/Emerging_Coded_Hate_Speech/data/raiders/reference_corpora/antisemitism_v1


## 2. Build and save, or reload

This cell scans the whole reference slice, matches anchors, selects contexts and saves:

- `reference_posts.jsonl`: original post metadata/HTML, readable text, source IDs and text hashes.
- `reference_occurrences.csv`: occurrence ID, post/thread IDs, anchor, matched text and exact offsets.
- `manifest.json`: source and file checksums, vocabulary, selection settings, counts and the unvalidated-reference assumption.

Every span satisfies `text[start:end] == matched_text`. The HTML-to-text logic matches the existing discovery preparation notebook. Nothing lowercases or removes stopwords from the saved text.

Rerunning loads the same saved references after checking their source and settings. It does not resample. Coverage is informational and does not block preparation.


In [2]:
reference_posts, reference_occurrences, coverage, manifest = prepare_references(
    REFERENCE_PATH, OUTPUT_DIR, anchors, CATEGORY,
    max_per_anchor=MAX_REFERENCES_PER_ANCHOR,
    random_seed=RANDOM_SEED,
)

print(manifest["scan_counts"])
print("Saved reference posts:", len(reference_posts))
print("Saved anchor occurrences:", len(reference_occurrences))


Scanned 2,000 reference threads; 468 occurrences.
Scanned 4,000 reference threads; 837 occurrences.
Scanned 6,000 reference threads; 1,447 occurrences.
Scanned 8,000 reference threads; 2,221 occurrences.
Scanned 10,000 reference threads; 2,741 occurrences.
Scanned 12,000 reference threads; 3,181 occurrences.
Scanned 14,000 reference threads; 3,459 occurrences.
Scanned 16,000 reference threads; 3,865 occurrences.
Scanned 18,000 reference threads; 4,233 occurrences.
Scanned 20,000 reference threads; 4,442 occurrences.
Scanned 22,000 reference threads; 4,617 occurrences.
Scanned 24,000 reference threads; 4,830 occurrences.
Scanned 26,000 reference threads; 5,172 occurrences.
Scanned 28,000 reference threads; 5,629 occurrences.
Scanned 30,000 reference threads; 6,034 occurrences.
Scanned 32,000 reference threads; 6,460 occurrences.
Saved 113 reference posts and 113 occurrences to /home/armand/MS_IA/IA717/project/Emerging_Coded_Hate_Speech/data/raiders/reference_corpora/antisemitism_v1
{'th

## 3. Anchor coverage (visualisation only)

The table compares all matches in the source with selected references. Occurrences, posts and threads are different counts: one post can contain several matches or anchors. Zero-coverage anchors remain visible. `repeated_text_posts` counts extra posts with equal text after case/whitespace normalization; this does not detect shared quotations or near copies.

The bars show the number of saved contexts per anchor. No labels or decisions need to be entered.


In [3]:
display(coverage.sort_values("posts", ascending=False).reset_index(drop=True))
print("Represented anchors:", int(coverage["selected_occurrences"].gt(0).sum()), "/", len(anchors))
print("Absent anchors:", coverage.loc[coverage["posts"].eq(0), "anchor"].tolist())

# HTML bars avoid adding a plotting dependency.
chart = coverage.sort_values("selected_occurrences", ascending=False)
maximum = max(int(chart["selected_occurrences"].max()), 1)
rows = []
for row in chart.itertuples(index=False):
    count = int(row.selected_occurrences)
    rows.append(
        f'<tr><td style="padding-right:12px">{escape(row.anchor)}</td>'
        f'<td style="width:300px"><div style="background:#16878b;height:14px;'
        f'width:{100 * count / maximum:.1f}%"></div></td><td>{count}</td></tr>'
    )
display(HTML("<b>Saved reference contexts per anchor</b><table>" + "".join(rows) + "</table>"))


,anchor,occurrences,posts,threads,distinct_normalized_texts,repeated_text_posts,selected_occurrences,selected_posts,selected_threads
0,kike,3586,3401,2042,3309,92,15,15,15
1,holocaust/gas chambers,2595,2129,1071,2066,63,15,15,15
2,holohoax,211,201,162,192,9,15,15,15
3,the goyim know,48,47,44,42,5,15,15,15
4,jewish power,22,21,21,21,0,15,15,15
5,great replacement,11,11,10,11,0,10,10,10
6,nazi symbols,8,8,7,8,0,7,7,7
7,protocols of the elders of zion,6,6,6,6,0,6,6,6
8,jew down,6,6,6,6,0,6,6,6
9,blood libel,5,5,5,5,0,5,5,5


Represented anchors: 13 / 20
Absent anchors: ['jew coup', 'jewish feature', 'jewish figure', 'jewish lighting', 'laughing jew', 'qanon', 'smirking merchant']


kike,,15
the goyim know,,15
holohoax,,15
holocaust/gas chambers,,15
jewish power,,15
great replacement,,10
nazi symbols,,7
protocols of the elders of zion,,6
jew down,,6
blood libel,,5
zionist occupied government,,2


## 4. Preview saved references

This is an optional preview, not an annotation task. Selected matches and their exact source text are already saved. Do not edit or reclean the saved text before embedding, because the offsets address that text.

Later discovery inputs must come from the separate discovery slice. The old pilot CSVs came from this first 1 GB and therefore overlap the reference source. Cross-partition copied text will still need checking when connecting the reference corpus to HMApp.


In [4]:
examples = reference_occurrences.merge(
    reference_posts[["post_id", "text"]], on="post_id", validate="many_to_one"
)
with pd.option_context("display.max_colwidth", 220):
    display(examples[["anchor", "post_id", "thread_id", "matched_text", "start", "end", "text"]].head(10))

print("Files:", [path.name for path in sorted(OUTPUT_DIR.iterdir())])
print("Reference text and spans are ready; no BERT vectors have been generated.")


,anchor,post_id,thread_id,matched_text,start,end,text
0,kike,79789590,79787361,kike,5,9,"That kike-puppet Reagan fucking up the entire world for his sweaty Jew master, Kissinger"
1,kike,79735862,79731602,kike,28,32,no u\n\nHappy freedom day you kike
2,kike,80403752,80402893,kike,92,96,>niggers respect masculinity in leaders\n>so they vote for a woman instead of a decrepit old kike\nactually makes sense to me
3,kike,79276065,79273864,Kike,128,132,It was probably revenge for Israel killing his little sister or something. Either way nothing of value was lost in both events. Kike's aern't human
4,kike,80620599,80613072,Kike,59,63,>implying all nations are the enemies of all other nations\nKike logic.
5,kike,79579147,79578051,kike,14,18,keep dreaming kike\n\nScotland leaving would be okay though because then the English don't have to finance their retardation and Scotland will become a 2nd Greece
6,kike,79744669,79742272,kike,0,4,kike spotted
7,kike,79745378,79742854,Kike,163,167,>The guy isnt a complete 100% image of the perfect savior messiah of the shining heroism\n\n>That means you should stop everything you're doing and elect Con Artist Kike or an Angry Warhawk Bull Dyke\n\n\nThis is wha...
8,kike,79835187,79834830,kike,143,147,>cute jewess\n\nYou fell into the kike trap. Every single one of those rats needs to die. Every. Single. One.\n\nThat would include your halfbreed kike kids.
9,kike,79861161,79852074,kike,33,37,would you be happy sat next to a kike?


Files: ['manifest.json', 'reference_occurrences.csv', 'reference_posts.jsonl']
Reference text and spans are ready; no BERT vectors have been generated.
